# Week 04 · 测试、Docker 与持续集成

单元测试聚焦小块纯业务逻辑；集成测试检验组件边界，例如 API 到数据库。Mock 应放在不稳定外部边界，不能把被测试的核心逻辑也替换成假的。日志服务于定位问题，不能记录密码、token 或完整隐私请求体。配置与代码分离，.env.example 只放示例，不放真实密钥。

Docker 镜像是可分发的文件系统和配置；容器是运行实例。COPY 的路径相对于构建上下文，.dockerignore 防止把密钥和大文件送入构建。容器内 127.0.0.1 指自身，Compose 服务之间用服务名通信。volume 让数据库数据跨容器重建存活。depends_on 不等于数据库立刻可用，需要健康检查或应用重连。

CI 每次提交自动装依赖、执行测试。下面生成可读的配置文件并在本机真实运行 unittest；生成 YAML 不是 GitHub Actions 已成功的证据。Docker 和 GitHub 流程需要你安装 Docker、连接仓库后再执行验证。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
from pathlib import Path
import unittest, logging, io, json
from unittest.mock import Mock

def create_task(title, repository):
    title = title.strip()
    if not title:
        raise ValueError("empty title")
    return repository.save({"title": title, "done": False})

class TaskTests(unittest.TestCase):
    def test_success(self):
        repository = Mock()
        repository.save.return_value = 42
        self.assertEqual(create_task("  learn CI ", repository), 42)
        repository.save.assert_called_once_with({"title":"learn CI", "done":False})
    def test_invalid_never_writes(self):
        repository = Mock()
        with self.assertRaises(ValueError): create_task(" ", repository)
        repository.save.assert_not_called()
    def test_storage_failure_propagates(self):
        repository = Mock()
        repository.save.side_effect = OSError("database unavailable")
        with self.assertRaises(OSError): create_task("test", repository)

output = io.StringIO()
result = unittest.TextTestRunner(stream=output, verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(TaskTests))
assert result.wasSuccessful()
print(output.getvalue())
folder = Path("week04-delivery"); folder.mkdir(exist_ok=True)
files = {
 "Dockerfile": "FROM python:3.13-slim\nWORKDIR /app\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY . .\nCMD [\"uvicorn\",\"main:app\",\"--host\",\"0.0.0.0\",\"--port\",\"8000\"]\n",
 ".dockerignore": ".git\n.venv\n.env\n__pycache__\n",
 "compose.yaml": "services:\n  api:\n    build: .\n    ports: ['127.0.0.1:8000:8000']\n    environment:\n      DATABASE_URL: postgresql+psycopg://app:example@db/app\n    depends_on:\n      db:\n        condition: service_healthy\n  db:\n    image: postgres:17\n    environment:\n      POSTGRES_USER: app\n      POSTGRES_PASSWORD: example\n      POSTGRES_DB: app\n    volumes: ['dbdata:/var/lib/postgresql/data']\n    healthcheck:\n      test: ['CMD-SHELL', 'pg_isready -U app']\n      interval: 5s\n      retries: 10\nvolumes:\n  dbdata:\n",
 "ci.yml": "name: tests\non: [push, pull_request]\njobs:\n  test:\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: '3.13'\n      - run: pip install -r requirements.txt\n      - run: python -m pytest -q\n",
}
for name, content in files.items():
    (folder/name).write_text(content, encoding="utf-8")
    print("配置草稿：", name)
print(json.dumps({"event":"unit_tests_completed", "count":result.testsRun}))

## 练习 / Exercises
把 Week 02 API 放入生成目录；补 requirements.txt 和健康检查，然后执行 docker compose up --build。不要把示例数据库密码用于公网。

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
# 配置静态检查只证明所需字段存在，不能冒充容器运行成功。
assert "127.0.0.1:8000:8000" in files["compose.yaml"]
assert ".env" in files[".dockerignore"]
print("本地单元测试真实通过；Docker/云端 CI 仍需实际运行验收。")